In [4]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import wavfile
from scipy import signal
from scipy import constants as const
from scipy import optimize
from scipy.interpolate import UnivariateSpline
import math
import warnings
from scipy.io.wavfile import WavFileWarning
from scipy.optimize import curve_fit

def importWAV(filename):
    warnings.filterwarnings("ignore", category=WavFileWarning)
    samplerate, rawData = wavfile.read(filename)
    
    time = np.linspace(0, rawData.shape[0]/samplerate, rawData.shape[0])   
    
    data = {'left':rawData[:, 0],'right':rawData[:, 1]}
    
    return time,data

def uncertainty(x, uX, y, uY):
    if x==0 or y==0:
        return 0
    return np.sqrt((uX/x)**2 + (uY/y)**2)

def MM_resistance_uncertainty(R):
    if R <= 99.99:
        return R * 1/100 + 0.03
    elif R <= 999.9:
        return R * 1/200 + 0.3
    elif R <= 9999:
        return R * 1/200 + 3
    elif R <= 99990:
        return R * 1/200 + 30
    elif R <= 999900:
        return R * 1/200 + 300
    else:
        return R * 3/200 + 3000

def gain_freq_dep(f):
    
    # Gain G_0 (unitless) - rounded based on one significant figure of the uncertainty
    G0 = 2170

    # Cutoff frequency f_c (Hz)
    fc = 9900

    Gf = G0 / np.sqrt(1 + (f/fc)**2)

    return Gf 

def aud_to_volt(data):

    slope = 107.0

    intercept  = -0.6 

    transformation = 0.001*(slope*data - intercept)

    return transformation

def JN_theoretical_value(R, uR, T, uT):

    theory, uTheory = 4 * const.k * R * T, 4 * const.k * R * T * uncertainty(R, uR, T, uT)
    
    return theory, uTheory

def percent_difference(value, theoretical):
    return np.abs(value - theoretical)/theoretical * 100

def gain_model(f, G0, fc):
    return G0 / np.sqrt(1 + (f/fc)**2)

# Define the model function for a line
def linear_model(x, m, b):
    return m * x + b

def amplitude_vs_frequency(freqs, amps, input_volt, frequencyHz, gain, u_gain, smooth=0, npts=2000):

    # Gain of amplifier -------------------------------------------------------------------------------------

    # --- Fit the data ---
    popt, pcov = curve_fit(gain_model, frequencyHz, gain, p0=[2200, 10000], sigma=u_gain, absolute_sigma=True)
    G0_fit, fc_fit = popt
    perr = np.sqrt(np.diag(pcov))  # uncertainties
    
    # --- Compute residuals and reduced chi-square ---
    residuals = gain - gain_model(frequencyHz, *popt)
    dof = len(gain) - len(popt)
    reduced_chi2 = np.sum((residuals / u_gain)**2) / dof
    
    # --- Print fitting information ---
    print("=== Fit Results ===")
    print(f"G = {G0_fit:.2f} ± {perr[0]:.2f}")
    print(f"fc = {fc_fit:.2f} kHz ± {perr[1]:.2f} kHz")
    print(f"Reduced Chi-square = {reduced_chi2:.2f}")
    print(f"Degrees of freedom = {dof}")
    
    # Frequency Dependence of Sound Card -------------------------------------------------------------------

    freqs = np.asarray(freqs)
    amps = np.asarray(aud_to_volt(amps))

    input_amps = np.full_like(amps, input_volt)

    idx = np.argsort(freqs)
    freqs_sorted = freqs[idx]
    amps_sorted = amps[idx] / (input_amps)

    spline = UnivariateSpline(freqs_sorted, amps_sorted, k=3, s=smooth)

    # Plotting ---------------------------------------------------------------------------------------------

    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(6,4), gridspec_kw={'height_ratios': [3,2]})

    plt.subplots_adjust(hspace=0.3)

    f_fit = np.linspace(50, 20000, 400)

    fgrid = np.linspace(freqs_sorted.min(), freqs_sorted.max(), npts)
    fit_vals = spline(fgrid)

    # Top: data + fit
    ax_top.text(-0.15, 1.1, '(a)', transform=ax_top.transAxes, fontsize=10, va='top', ha='left')
    ax_top.errorbar(frequencyHz, gain, yerr=u_gain, fmt='o', capsize=3, label="Data")
    ax_top.plot(f_fit, gain_model(f_fit, *popt), 'r-', label=f"Fit: G0={G0_fit:.1f}, fc={fc_fit:.2f} Hz")
    ax_top.set_ylabel("Gain")
    ax_top.set_xlim(-500, 20500)
    ax_top.set_ylim(np.min(gain) - 100, np.max(gain) + 100)

    ax_bot.text(-0.15, 1.1, '(b)', transform=ax_bot.transAxes, fontsize=10, va='top', ha='left')
    ax_bot.scatter(freqs_sorted, amps_sorted, label="Data", color="blue")
    ax_bot.plot(fgrid, fit_vals, label="Spline Fit", linewidth=2, color="red")
    ax_bot.set_xlim(-500, 20500)
    ax_bot.set_ylim(np.min(amps_sorted) - 0.01, np.max(amps_sorted) + 0.01)
    plt.xlabel(fr"Frequency [Hz]")
    plt.ylabel(r"$\text{V}_\text{SC} \; / \; \text{V}_\text{in}$")
    #plt.title("Digital to Voltage Frequency-Dependent Correction Factor vs Frequency — Spline Fit")
    plt.show()

    fig.tight_layout()

    fig.savefig("calibration.pdf")

    return spline

def JN_Analysis(filename, fs, T, R, calibration_spline, start_freq = 0, end_freq = -1, nperseg=2**8, semilog = False, verbose = False, plot_correlations = False):

    # Import .WAV file and extract signal data CH1 and CH2
    time, data = importWAV(filename)

    voltage_data_left = aud_to_volt(data['left'])

    voltage_data_right = aud_to_volt(data['right'])

    # Calculate Theoretical JN value with uncertainty from resistance measurement uncertainty, temperature measurement uncertainty
    JN_theory = JN_theoretical_value(R, MM_resistance_uncertainty(R), T, 0.1)

    # Perform PSD
    f, Pxx_den = signal.welch(voltage_data_left, fs, nperseg=nperseg)

    # Perform CSD and take real portion of signal
    f, Pxy_den = signal.csd(voltage_data_left, voltage_data_right, fs, nperseg=nperseg)

    Pxy_den = Pxy_den.real

    # Calibration factor for frequency-dependence of sound card
    cal_factor = 1/calibration_spline(f)

    # Gain adjustment factor for gain calibration of amplifier
    gain_adj = gain_freq_dep(f)

    # Gain-adjusted PSD (Pxx) and CSD (Pxy)
    Pxx_den_gain_adj = Pxx_den / (gain_adj**2 ) * cal_factor**2

    Pxy_den_gain_adj = Pxy_den / (gain_adj**2 ) * cal_factor**2

    # Bin size for spectral densities
    binsize = f[1]-f[0]

    # Calculate start/end indices from the start/end frequencies argument with error handling
    start_idx = max(0, int(start_freq/binsize))
    end_idx = min(int(fs/2), int(end_freq/binsize))

    if start_idx >= end_idx:
        raise ValueError(f"Start index {start_idx} >= end index {end_idx}")
        
    # Calculate average raw PSD noise power density
    noisepowerden_PSD_Raw = np.average(Pxx_den[start_idx: end_idx])

    # Calculate average gain-adjusted PSD noise power density and uncertainty
    
    noisepowerden_PSD = np.average(Pxx_den_gain_adj[start_idx: end_idx])

    correlate_PSD_unnormalized = signal.correlate(Pxx_den_gain_adj[start_idx: end_idx], Pxx_den_gain_adj[start_idx: end_idx], mode="full")

    correlate_PSD = correlate_PSD_unnormalized / np.max(correlate_PSD_unnormalized)

    mid_PSD = len(correlate_PSD)//2

    neg_zeros_PSD = np.where(correlate_PSD[:mid_PSD][::-1] <= 0)[0]
    neg_idx_PSD = mid_PSD - neg_zero_PSD[0] if len(neg_zeros_PSD) > 0 else 0
    
    pos_zeros_PSD = np.where(correlate_PSD[mid_PSD:] <= 0)[0]
    pos_idx_PSD = mid_PSD + pos_zero_PSD[0] if len(pos_zeros_PSD) > 0 else len(correlate_PSD)
    
    tau_f_PSD = np.sum(correlate_PSD[neg_idx_PSD:pos_idx_PSD]) * binsize
    
    N_eff_PSD = (len(Pxx_den_gain_adj[start_idx: end_idx]) * binsize) / tau_f_PSD

    sigma_PSD = np.std(Pxx_den_gain_adj[start_idx: end_idx]) / np.sqrt(N_eff_PSD)

    if plot_correlations:
        fig_corr, (ax1_corr, ax2_corr) = plt.subplots(1, 2, figsize=(12,4))
        lags_PSD = np.arange(-len(Pxx_den_gain_adj[start_idx: end_idx])+1, len(Pxx_den_gain_adj[start_idx: end_idx]))
        ax1_corr.plot(lags_PSD, correlate_PSD)
        ax1_corr.set_xlabel(f"Frequency Bins [{binsize} Hz/bin]")
        ax1_corr.set_ylabel(f"Normalized Correlations")

    # Calculate average PSD noise power density and uncertainty
    noisepowerden_CSD = np.average(Pxy_den_gain_adj[start_idx: end_idx])

    correlate_CSD_unnormalized = signal.correlate(Pxy_den_gain_adj[start_idx: end_idx], Pxy_den_gain_adj[start_idx: end_idx], mode="full")

    correlate_CSD = correlate_CSD_unnormalized / np.max(correlate_CSD_unnormalized)

    mid_CSD = len(correlate_CSD)//2

    neg_zeros_CSD = np.where(correlate_CSD[:mid_CSD][::-1] <= 0)[0]
    neg_idx_CSD = mid_CSD - neg_zero_CSD[0] if len(neg_zeros_CSD) > 0 else 0
    
    pos_zeros_CSD = np.where(correlate_CSD[mid_CSD:] <= 0)[0]
    pos_idx_CSD = mid_CSD + pos_zero_CSD[0] if len(pos_zeros_CSD) > 0 else len(correlate_CSD)
    
    tau_f_CSD = np.sum(correlate_CSD[neg_idx_CSD:pos_idx_CSD]) * binsize
    
    N_eff_CSD = (len(Pxy_den_gain_adj[start_idx: end_idx]) * binsize) / tau_f_CSD

    sigma_CSD = np.std(Pxy_den_gain_adj[start_idx: end_idx]) / np.sqrt(N_eff_CSD)

    if plot_correlations:
        lags_CSD = np.arange(-len(Pxy_den_gain_adj[start_idx: end_idx])+1, len(Pxy_den_gain_adj[start_idx: end_idx]))
        ax2_corr.plot(lags_CSD, correlate_CSD)
        ax2_corr.set_xlabel(f"Frequency Bins [{binsize} Hz/Bin]")
        ax2_corr.set_ylabel(f"Normalized Correlations")

    if verbose:
        print(fr"Analysis for Resistance R = {R} ± {MM_resistance_uncertainty(R)}Ω")
        print("Bin size is",binsize)
        print(f"Average noise power density for PSD over the stated range is {noisepowerden_PSD} ± {sigma_PSD}")
        print(f"Average noise power density for CSD over the stated range is {noisepowerden_CSD} ± {sigma_CSD}")
        print(f"Range examined is from f={start_idx*binsize} Hz to f={end_idx*binsize} Hz")
        print(f"Theoretic JN is {JN_theory[0]} ± {JN_theory[1]} V**2/Hz")
        print(f"Percent difference between Data and Theoretical for Gain Adjusted PSD = {percent_difference(noisepowerden_PSD, JN_theory[0])}%")
        print(f"Agreement: {np.abs(noisepowerden_PSD - JN_theory[0]) < (sigma_PSD + JN_theory[1])}")
        print(f"Percent difference between Data and Theoretical for Gain Adjusted CSD = {percent_difference(noisepowerden_CSD, JN_theory[0])}%")
        print(f"Agreement: {np.abs(noisepowerden_CSD - JN_theory[0]) < (sigma_CSD + JN_theory[1])}")
        print(f"\n\n")

    freq_range = f[start_idx: end_idx]

    avg_noisepowerden_PSD_Raw = np.full_like(freq_range, noisepowerden_PSD_Raw)
    avg_noisepowerden_PSD = np.full_like(freq_range, noisepowerden_PSD)
    avg_noisepowerden_CSD = np.full_like(freq_range, noisepowerden_CSD)

    
    fig, (ax1, ax2, ax3) = plt.subplots(3,1, figsize=(6,6))

    ax1.text(-0.1, 1.1, '(a)', transform=ax1.transAxes, fontsize=10, va='top', ha='left')
    #ax1.set_xlabel(rf"Frequency [Hz]", fontsize=10)
    ax1.set_ylabel(r"PSD$_\text{Raw}$ [V$^2$/Hz]")
    if semilog:
        ax1.semilogy(f, Pxx_den, label='Data - Raw', color='gray', alpha=0.3, zorder=4)
        ax1.semilogy(f[start_idx:end_idx], Pxx_den[start_idx:end_idx], label='Analyzed Data - Raw', color='blue', alpha=0.5, zorder=4)
    else:
        ax1.plot(f, Pxx_den, label='Data - Raw', color='gray', alpha=0.3, zorder=4)
        ax1.plot(f[start_idx:end_idx], Pxx_den[start_idx:end_idx], label='Analyzed Data - Raw', color='blue', alpha=0.6, zorder=4)
    ax1.fill_between(f, JN_theory[0] - JN_theory[1], JN_theory[0] + JN_theory[1], color='r', alpha=0.6, label='Theory ± Uncertainty', zorder=5)
    ax1.plot(freq_range, avg_noisepowerden_PSD_Raw, label = 'Average Raw PSD on Reduced Band', color='k', zorder=4)
    #ax1.legend()

    ax2.text(-0.1, 1.1, '(b)', transform=ax2.transAxes, fontsize=12, va='top', ha='left')
    #ax2.set_xlabel(rf"Frequency [Hz]", fontsize=10)
    ax2.set_ylabel(r"PSD$_\text{GA}$ [V$^2$/Hz]")
    if semilog:
        ax2.semilogy(f, Pxx_den_gain_adj, label='Gain Adjusted PSD', color='gray', alpha=0.3, zorder=4)
        ax2.semilogy(f[start_idx:end_idx], Pxx_den_gain_adj[start_idx:end_idx], label='Gain Adjusted PSD - Reduced Band', color='blue', alpha=0.5, zorder=4)
    else:
        ax2.plot(f, Pxx_den_gain_adj, label='Gain Adjusted PSD', color='gray', alpha=0.3, zorder=4)
        ax2.plot(f[start_idx:end_idx], Pxx_den_gain_adj[start_idx:end_idx], label='Gain Adjusted PSD - Reduced Band', color='blue', alpha=0.5, zorder=4)
    ax2.fill_between(f, JN_theory[0] - JN_theory[1], JN_theory[0] + JN_theory[1], color='r', alpha=0.6, label='Theory ± Uncertainty', zorder=5)
    ax2.plot(freq_range, avg_noisepowerden_PSD, label = 'Average Gain Adjusted PSD on Reduced Band', color='k', zorder=4)
    #ax2.legend()

    ax3.text(-0.1, 1.1, '(c)', transform=ax3.transAxes, fontsize=12, va='top', ha='left')
    ax3.set_xlabel(rf"Frequency [Hz]", fontsize=10)
    ax3.set_ylabel(r"CSD$_\text{GA}$ [V$^2$/Hz]")
    if semilog:
        ax3.semilogy(f, Pxy_den_gain_adj, label='Gain Adjusted CSD', color='gray', alpha=0.3, zorder=4)
        ax3.semilogy(f[start_idx:end_idx], Pxy_den_gain_adj[start_idx:end_idx], label='Gain Adjusted CSD', color='blue', alpha=0.6, zorder=4)
    else:
        ax3.plot(f, Pxy_den_gain_adj, label='Gain Adjusted CSD', color='gray', alpha=0.3, zorder=4)
        ax3.plot(f[start_idx:end_idx], Pxy_den_gain_adj[start_idx:end_idx], label='Gain Adjusted CSD', color='blue', alpha=0.6, zorder=4)
    ax3.fill_between(f, JN_theory[0] - JN_theory[1], JN_theory[0] + JN_theory[1], color='r', alpha=0.6, label='Theory ± Uncertainty', zorder=5)
    ax3.plot(freq_range, avg_noisepowerden_CSD, label = 'Average Gain Adjusted CSD on Reduced Band', color='k', zorder=4)
    #ax3.legend()

    fig.tight_layout(pad=0.75)

    plt.show()

    fig.savefig(f"{filename[:-4]}.pdf")

    return [R, noisepowerden_PSD, sigma_CSD, noisepowerden_CSD, sigma_CSD, JN_theory[0], JN_theory[1]]

def plot_JN_vs_Resistance(R, noisepowerden_PSD, uNoisepowerden_PSD, noisepowerden_CSD, uNoisepowerden_CSD, JN_theory, uJN_theory, T, uT):

    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(5,4), gridspec_kw={'height_ratios': [3,2]})

    # Perform the curve fit for PSD values with weights
    popt_PSD, pcov_PSD = optimize.curve_fit(linear_model, R, noisepowerden_PSD, sigma=uNoisepowerden_PSD, absolute_sigma=True)
    m_PSD, b_PSD = popt_PSD
    m_err_PSD, b_err_PSD = np.sqrt(np.diag(pcov_PSD))

    # Perform the curve fit for PSD values with weights
    popt_CSD, pcov_CSD = optimize.curve_fit(linear_model, R, noisepowerden_CSD, sigma=uNoisepowerden_CSD, absolute_sigma=True)
    m_CSD, b_CSD = popt_CSD
    m_err_CSD, b_err_CSD = np.sqrt(np.diag(pcov_CSD))

    R_fit = np.linspace(min(R), max(R), 200)
    noisepowerden_PSD_fit = linear_model(R_fit, m_PSD, b_PSD)
    noisepowerden_CSD_fit = linear_model(R_fit, m_CSD, b_CSD)

    # --- Chi-square analysis for PSD ---
    model_residuals_PSD = noisepowerden_PSD - linear_model(R, m_PSD, b_PSD)
    chi2_PSD = np.sum((model_residuals_PSD / uNoisepowerden_PSD) ** 2)
    
    N = len(R)      # number of data points
    k_PSD = len(popt_PSD)        # number of fit parameters
    dof_PSD = N - k_PSD          # degrees of freedom
    chi2_PSD_red = chi2_PSD / dof_PSD

    # --- Chi-square analysis for PSD ---
    model_residuals_CSD = noisepowerden_CSD - linear_model(R, m_CSD, b_CSD)
    chi2_CSD = np.sum((model_residuals_CSD / uNoisepowerden_CSD) ** 2)
    
    k_CSD = len(popt_CSD)        # number of fit parameters
    dof_CSD = N - k_CSD          # degrees of freedom
    chi2_CSD_red = chi2_CSD / dof_CSD

    m_theory = 4 * const.k * T
    m_theory_err = 4 * const.k * uT
    
    # Print results
    print(rf"Theory Slope(m_theory): {m_theory} ± {m_theory_err}")
    
    print(f"PSD Slope (m_PSD): {m_PSD} ± {m_err_PSD}")
    #print(f"Agrees with theory?: {np.abs(m_PSD - m_theory)<(m_err_PSD + m_theory_err)}")
    print(f"PSD Intercept (b_PSD): {b_PSD} ± {b_err_PSD}")
    print(f"PSD Reduced Chi-square: {chi2_PSD_red}")

    # Print results
    print(f"CSD Slope (m)_CSD: {m_CSD} ± {m_err_CSD}")
    #print(f"Agrees with theory?: {np.abs(m_CSD - m_theory)<(m_err_CSD + m_theory_err)}")
    print(f"CSD Intercept (b_CSD): {b_CSD} ± {b_err_CSD}")
    print(f"CSD Reduced Chi-square: {chi2_CSD_red}")

    # Check agreement using only experimental uncertainty
    agrees_PSD = np.abs(m_PSD - m_theory) < m_err_PSD
    agrees_CSD = np.abs(m_CSD - m_theory) < m_err_CSD
    
    # Percent differences
    pct_diff_PSD = 100 * np.abs(m_PSD - m_theory) / m_theory
    pct_diff_CSD = 100 * np.abs(m_CSD - m_theory) / m_theory
    
    print(f"PSD agrees with theory? {agrees_PSD}, Percent difference = {pct_diff_PSD:.2f}%")
    print(f"CSD agrees with theory? {agrees_CSD}, Percent difference = {pct_diff_CSD:.2f}%")

    residuals_PSD = noisepowerden_PSD - JN_theory
    
    residuals_PSD_err = np.sqrt(uNoisepowerden_PSD**2 + uJN_theory**2)

    residuals_CSD = noisepowerden_CSD - JN_theory

    residuals_CSD_err = np.sqrt(uNoisepowerden_CSD**2 + uJN_theory**2)

    ax_top.text(-0.1, 1.1, '(a)', transform=ax_top.transAxes, fontsize=10, va='top', ha='left')
    ax_top.errorbar(R, noisepowerden_PSD, np.sqrt(uNoisepowerden_PSD**2 + uJN_theory**2), fmt='o', label="Average PSD", color='black')
    ax_top.plot(R_fit, noisepowerden_PSD_fit, color='black', linestyle='--')
    ax_top.errorbar(R, noisepowerden_CSD, np.sqrt(uNoisepowerden_CSD**2 + uJN_theory**2), fmt='o', label="Average CSD", color='blue')
    ax_top.plot(R_fit, noisepowerden_CSD_fit, color='blue', linestyle='--')
    ax_top.plot(R, JN_theory, label="Theoretical Johnson Noise", linestyle='--', color='red')
    #ax_top.set_xscale('log')
    #ax_top.set_yscale('log')
    #plt.title('Cross Spectral Density vs Resistance, log scale', fontsize=14)
    #ax_top.set_xlabel(r'Resistance [$\Omega$]', fontsize=10)
    ax_top.set_ylabel('SD [V$^2$/Hz]', fontsize=10)

    ax_bot.text(-0.1, 1.1, '(b)', transform=ax_bot.transAxes, fontsize=10, va='top', ha='left')
    ax_bot.errorbar(R, residuals_PSD, residuals_PSD_err, fmt='o', label="PSD vs Theory Residuals", color='black')
    ax_bot.errorbar(R, residuals_CSD, residuals_CSD_err, fmt='o', label="CSD vs Theory Residuals", color='blue')
    ax_bot.hlines(0, R.min()*0.8, R.max()*1.2, colors='red', linestyle='--')
    ax_bot.set_xlabel("Resistance [Ω]")
    ax_bot.set_ylabel("Residuals")    

    fig.tight_layout(pad=0.75)

    plt.show()

    fig.savefig("JNvsR.pdf")

In [5]:
##### --- Data ---
frequency = np.array([
    1.000, 0.900, 0.800, 0.700, 0.600, 0.500, 0.400, 0.300, 0.200, 0.100, 0.050,
    2.000, 3.000, 4.000, 5.000, 6.000, 7.000, 8.000, 9.000, 10.000, 11.000, 12.000,
    13.000, 14.000, 15.000, 16.000, 17.000, 18.000, 19.000, 20.000
])

frequencyHz = np.array([
    1000, 900, 800, 700, 600, 500, 400, 300, 200, 100, 50,
    2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000,
    10000, 11000, 12000, 13000, 14000, 15000, 16000,
    17000, 18000, 19000, 20000
])

gain = np.array([
    2161, 2163, 2166, 2167, 2168, 2168, 2169, 2168, 2165, 2148, 2085,
    2132, 2084, 2022, 1948, 1868, 1785, 1700, 1618, 1538, 1452, 1388,
    1319, 1254, 1192, 1136, 1083, 1033, 987, 943
])

u_gain = np.array([
    43, 43, 44, 44, 44, 44, 44, 44, 43, 43, 42,
    43, 42, 41, 39, 38, 36, 34, 33, 31, 29, 28,
    27, 25, 24, 23, 22, 21, 20, 19
])

In [ ]:
frequency_in = np.linspace(500, 15000, 30)
amplitudes = np.zeros(30)

for i in range(0,30):
    filename = "../raw_data/" + str(frequency_in[i])[:-2] + "Hz.wav"
    time, data = importWAV(filename)
    amplitudes[i] = np.abs(np.max(data['left']) - np.min(data['left']))/2

FileNotFoundError: [Errno 2] No such file or directory: '500Hz.wav'

In [ ]:
v_RMS = 54.18 #mV
v_input = v_RMS*np.sqrt(2)

calibration_spline = amplitude_vs_frequency(frequency_in, amplitudes, v_input/1000, frequencyHz, gain, u_gain)

In [ ]:
file_data = [["JN_R0_1min.wav", 0], ["JN_R100Ohm_1min.wav", 99.4], ["JN_R1kOhm_1min.wav", 987], ["JN_R4kOhm_1min.wav", 3912], ["JN_R10kOhm_1min.wav", 9870], ["JN_R12kOhm_1min.wav", 11840], ["JN_R100kOhm_1min.wav", 99500], ["JN_R1MOhm_1min.wav", 994000]]

fs = 48000

n=2**12

start_freq = 500
end_freq = 15000

noisepowerden_data = np.zeros((len(file_data),7))

# Estimated class temperature in Kelvin
T = 22.1 + 273.2 #K

In [ ]:
for i in range(1, 8):
    noisepowerden_data[i] =  JN_Analysis(file_data[i][0], fs, T, file_data[i][1], calibration_spline, 500, 15500, n, verbose=True)

In [ ]:
data_start = 1
data_end = 6

plot_JN_vs_Resistance(noisepowerden_data[data_start:data_end, 0], noisepowerden_data[data_start:data_end, 1], noisepowerden_data[data_start:data_end, 2], noisepowerden_data[data_start:data_end, 3], noisepowerden_data[data_start:data_end, 4],noisepowerden_data[data_start:data_end, 5],noisepowerden_data[data_start:data_end, 6], T, 0.1)